In [1]:
import pandas as pd
import os
from tqdm import tqdm

In [2]:
df_all = pd.read_csv("../datasets/schaefcomb_Wang2023Simple_all.tsv", sep = "\t")

In [3]:
print(df_all.diagnosis.value_counts())
print(df_all.site_id.value_counts())

diagnosis
CONTROL    1135
Autism      863
SCHZ         96
BIPOLAR      49
ADHD         40
Name: count, dtype: int64
site_id
ds000030          261
ABIDEII-KKI_1     211
NYU               184
USM               101
ABIDEII-OHSU_1     91
ABIDEII-NYU_1      78
ABIDEII-GU_1       74
UCLA_1             73
ds004302           71
MAX_MUN            57
PITT               57
KKI                55
ABIDEII-IP_1       55
ABIDEII-SDSU_1     55
YALE               55
ABIDEII-BNI_1      54
ABIDEII-EMC_1      54
TRINITY            49
ABIDEII-ONRC_2     48
ABIDEII-TCD_1      42
ABIDEII-IU_1       40
CALTECH            38
ABIDEII-ETHZ_1     37
SDSU               36
OLIN               36
LEUVEN_2           35
ABIDEII-USM_1      32
ABIDEII-UCD_1      32
ABIDEII-UCLA_1     32
SBL                30
LEUVEN_1           29
OHSU               28
ABIDEII-KUL_3      27
UCLA_2             26
Name: count, dtype: int64


In [5]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, ParameterGrid, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.base import BaseEstimator, TransformerMixin

from tqdm import tqdm

# ============================================================
# 0) OPTIONS GLOBALES
# ============================================================

# Méthode de correction des confounds sur les connectomes
# "none"        : pas de correction de confounds
# "regression"  : régression (age, sexe)
# "combat"      : ComBat (site comme batch, age & sexe comme covariables)
CONF_METHOD = "none"   # "none" / "regression" / "combat"

# PCA en entrée du classif
USE_PCA = True
N_PCS   = 400

# Cross-validation
N_SPLITS = 10  # k-fold

# Grid de recherche d'hyperparamètres
GRID_C_VALUES        = [1]
GRID_PENALTIES       = ["l2"]
GRID_USE_AGE_SEX     = [False]


# ============================================================
# 1) SÉLECTION DES SUJETS & PRÉPARATION DE LA CIBLE
#    → AUTISM vs CONTROL, tous les sites SAUF ds000030 et ds004302
# ============================================================

# On garde uniquement CONTROL vs Autism, en excluant ds000030 et ds004302
mask_data = (
    df_all["diagnosis"].isin(["CONTROL", "Autism"])
    & ~df_all["site_id"].isin(["ds000030", "ds004302"])
)

df_data = df_all.loc[mask_data].copy()
print("Dataset AUTISM vs CONTROL (sites ≠ ds000030, ds004302) :", df_data.shape)
print(df_data["diagnosis"].value_counts())
print(df_data["site_id"].value_counts())

# Cible binaire : 1 = Autism, 0 = CONTROL
y = (df_data["diagnosis"] == "Autism").astype(int)

# Encodage sexe (0/1)
df_data["gender_num"] = (df_data["gender"] == "M").astype(float)


# ============================================================
# 2) CONNECTOMES : SÉLECTION + IMPUTATION (fit sur tout df_data)
# ============================================================

# Colonnes de connectome : celles qui commencent par "corr_"
corr_cols_all = [c for c in df_all.columns if c.startswith("corr_")]

# On retire les colonnes 100% NaN sur ce sous-ensemble df_data
nan_all_data = df_data[corr_cols_all].isna().all()
corr_cols = nan_all_data[~nan_all_data].index.tolist()
print(f"Colonnes corr_ retenues : {len(corr_cols)}")

# Matrice brute connectomes
X_corr_raw = df_data[corr_cols].copy()

# Imputation des NaN par la moyenne (fit sur df_data uniquement)
corr_imputer = SimpleImputer(strategy="mean")
X_corr_imp = corr_imputer.fit_transform(X_corr_raw)

# On retransforme en DataFrame pour garder index / noms
X_corr_imp = pd.DataFrame(X_corr_imp, index=df_data.index, columns=corr_cols)


# ============================================================
# 3) CORRECTION POUR LES VARIABLES CONFONDANTES
#    (Age, Sexe, et éventuellement SITE avec ComBat)
# ============================================================

def residualize_confounds_single(X, conf):
    """
    Régression linéaire multi-sortie :
        X = conf * B + erreur
    On ajuste B sur TOUT df_data (pas de split explicite),
    puis on enlève conf*B pour obtenir les résidus.
    """
    # On ajoute un intercept
    C = np.column_stack([np.ones(len(conf)), conf.values])

    # B = (C^T C)^-1 C^T X  (multi-feature)
    B = np.linalg.pinv(C).dot(X.values)

    # prédictions
    X_hat = C.dot(B)

    # résidus = données "déconfondées"
    X_resid = X.values - X_hat

    X_resid = pd.DataFrame(X_resid, index=X.index, columns=X.columns)
    return X_resid


if CONF_METHOD == "regression":
    # On corrige les connectomes pour Age + Sexe
    conf = df_data[["age", "gender_num"]]
    X_corr_corr = residualize_confounds_single(X_corr_imp, conf)
    print("Correction par régression (age, sexe) appliquée.")

elif CONF_METHOD == "combat":
    # ⚠️ Nécessite : pip install neuroHarmonize
    from neuroHarmonize import harmonizationLearn, harmonizationApply

    X_corr_all_imp = X_corr_imp.values

    covars_all = pd.DataFrame({
        "SITE":  df_data["site_id"].values,
        "AGE":   df_data["age"].values,
        "SEX_M": df_data["gender_num"].values,
    })

    print("Apprentissage ComBat pour corriger l'effet SITE...")
    combat_model, X_corr_all_adj = harmonizationLearn(
        X_corr_all_imp,
        covars_all,
        smooth_terms=["AGE"]
    )

    X_corr_corr = pd.DataFrame(
        X_corr_all_adj,
        index=df_data.index,
        columns=corr_cols
    )
    print("Correction ComBat (site) appliquée.")

else:
    # Pas de correction de confounds sur les connectomes
    X_corr_corr = X_corr_imp
    print("Aucune correction de confounds appliquée sur les connectomes.")


# ============================================================
# 4) DESIGN FINAL POUR LE CLASSIFIEUR
# ============================================================

# On concatène connectomes + age + sexe ; l'inclusion effective
# de age/gender_num sera décidée par le grid via un transformer
X_final = pd.concat(
    [X_corr_corr,
     df_data[["age", "gender_num"]]],
    axis=1
)

print("Shape X_final :", X_final.shape)
print("len(y)        :", len(y))


# ============================================================
# 5) TRANSFORMER POUR (DÉ)INCLURE age & gender_num
# ============================================================

class AgeSexSelector(BaseEstimator, TransformerMixin):
    """
    Si use_age_sex = True  -> garde toutes les colonnes telles quelles.
    Si use_age_sex = False -> retire 'age' et 'gender_num' (si présents).
    """
    def __init__(self, use_age_sex=True):
        self.use_age_sex = use_age_sex

    def fit(self, X, y=None):
        self._has_age = "age" in X.columns
        self._has_gender = "gender_num" in X.columns
        return self

    def transform(self, X):
        X_tr = X
        if not self.use_age_sex:
            cols_to_drop = []
            if self._has_age:
                cols_to_drop.append("age")
            if self._has_gender:
                cols_to_drop.append("gender_num")
            if len(cols_to_drop) > 0:
                X_tr = X_tr.drop(columns=cols_to_drop)
        return X_tr


# ============================================================
# 6) PIPELINE PCA + LOGISTIC REGRESSION
# ============================================================

feature_steps = [
    ("selector", AgeSexSelector()),      # on décidera via le grid: avec / sans age+sexe
    ("scaler", StandardScaler())
]

if USE_PCA:
    feature_steps.append(
        ("pca", PCA(n_components=N_PCS, random_state=42))
    )

feature_pipeline = Pipeline(feature_steps)

logreg = LogisticRegression(
    penalty="l2",          # sera écrasé par le grid
    solver="lbfgs",         # supporte l1 et l2
    C=1.0,                 # sera écrasé par le grid
    class_weight="balanced",
    max_iter=5000,
    n_jobs=-1,
    random_state=42
)

clf = Pipeline([
    ("features", feature_pipeline),
    ("logreg", logreg)
])


# ============================================================
# 7) GRID SEARCH MANUEL + TQDM
# ============================================================

param_grid = {
    "logreg__C": GRID_C_VALUES,
    "logreg__penalty": GRID_PENALTIES,
    "features__selector__use_age_sex": GRID_USE_AGE_SEX,
}

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

grid = list(ParameterGrid(param_grid))
print(f"\nNombre de combinaisons d'hyperparamètres : {len(grid)}")

best_score = -np.inf
best_params = None

print("\n=== Lancement du grid search avec tqdm ===")
for params in tqdm(grid):
    # Appliquer les hyperparamètres au pipeline
    clf.set_params(**params)

    # CV sur ces hyperparamètres
    scores = cross_val_score(
        clf,
        X_final,
        y,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    mean_score = scores.mean()

    if mean_score > best_score:
        best_score = mean_score
        best_params = params

print("\n=== Meilleurs hyperparamètres trouvés ===")
print(best_params)
print(f"Best AUC ROC (moyenne CV) : {best_score:.3f}")

# Refit du meilleur modèle sur toutes les données
best_clf = clf.set_params(**best_params)
best_clf.fit(X_final, y)

# Ensuite, tu peux utiliser best_clf pour prédire sur un jeu de test externe si tu en as un :
# y_proba_test = best_clf.predict_proba(X_test)[:, 1]
# y_pred_test  = best_clf.predict(X_test)
# print(classification_report(y_test, y_pred_test))


Dataset AUTISM vs CONTROL (sites ≠ ds000030, ds004302) : (1851, 93967)
diagnosis
CONTROL    988
Autism     863
Name: count, dtype: int64
site_id
ABIDEII-KKI_1     211
NYU               184
USM               101
ABIDEII-OHSU_1     91
ABIDEII-NYU_1      78
ABIDEII-GU_1       74
UCLA_1             73
MAX_MUN            57
PITT               57
KKI                55
ABIDEII-IP_1       55
ABIDEII-SDSU_1     55
YALE               55
ABIDEII-BNI_1      54
ABIDEII-EMC_1      54
TRINITY            49
ABIDEII-ONRC_2     48
ABIDEII-TCD_1      42
ABIDEII-IU_1       40
CALTECH            38
ABIDEII-ETHZ_1     37
SDSU               36
OLIN               36
LEUVEN_2           35
ABIDEII-USM_1      32
ABIDEII-UCD_1      32
ABIDEII-UCLA_1     32
SBL                30
LEUVEN_1           29
OHSU               28
ABIDEII-KUL_3      27
UCLA_2             26
Name: count, dtype: int64
Colonnes corr_ retenues : 93096
Aucune correction de confounds appliquée sur les connectomes.
Shape X_final : (1851, 93098)
l

100%|██████████| 1/1 [01:11<00:00, 71.27s/it]



=== Meilleurs hyperparamètres trouvés ===
{'features__selector__use_age_sex': False, 'logreg__C': 1, 'logreg__penalty': 'l2'}
Best AUC ROC (moyenne CV) : 0.751


,steps,"[('features', ...), ('logreg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('selector', ...), ('scaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,use_age_sex,False
,copy,True
,with_mean,True
